# Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Constants

In [ ]:
db_path = "../data/sqlite/data.db"
datasets_path = "../data/raw/"

# Analysis

In [ ]:
db = sqlite3.connect(db_path)

## Load Data

In [ ]:
def load_all_csv_in_db(root_path=datasets_path, chunksize=100000, db=db, table_name=None):
    initial_table_name = False
    if table_name:
        initial_table_name = True
    
    for item in os.listdir(root_path):
        full_path = os.path.join(root_path, item)
        
        if os.path.isdir(full_path):
            # If it's a directory, call recursively
            load_all_csv_in_db(full_path + "/", chunksize, db)
        elif item.endswith('.csv'):
            # If it's a CSV file, process directly
            print(f"Loading {full_path}...")
            
            try:
                # Read a sample to get the columns
                sample = pd.read_csv(full_path, nrows=1, low_memory=False)
                if initial_table_name == False:
                    table_name = os.path.splitext(item)[-2].replace('-', '_').replace(' ', '_')
                
                # Create table with the columns
                columns = ', '.join([f'"{col}" TEXT' for col in sample.columns])
                db.execute(f"CREATE TABLE IF NOT EXISTS {table_name} ({columns})")
                db.commit()
                
                # Insert data in chunks
                for chunk in pd.read_csv(full_path, chunksize=chunksize, low_memory=False):
                    chunk.to_sql(table_name, db, if_exists='append', index=False)
                    db.commit()
                
                print(f"  ✓ Completed: {item}")
            except Exception as e:
                print(f"  ✗ Error loading {item}: {e}")
                continue

In [ ]:
load_all_csv_in_db(root_path="../data/raw/CIC_APT_IIoT_2024/", table_name="CIC_APT_IIoT_2024")

Loading ../data/raw/CIC_APT_IIoT_2024/phase1_NetworkData.csv...
  ✓ Completed: phase1_NetworkData.csv
Loading ../data/raw/CIC_APT_IIoT_2024/phase2_NetworkData.csv...
  ✓ Completed: phase2_NetworkData.csv


## Statistics

In [ ]:
"""
sns.set_style("whitegrid")

# plot
fig, ax = plt.subplots(figsize=(10, 6))

counts = df["label"].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax, edgecolor="white", linewidth=0.7)

ax.set_xlabel("TYPE")
ax.set_ylabel("COUNT")
ax.tick_params(axis="x", labelrotation=45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

plt.tight_layout()
plt.show()
"""

'\nsns.set_style("whitegrid")\n\n# plot\nfig, ax = plt.subplots(figsize=(10, 6))\n\ncounts = df["label"].value_counts()\nsns.barplot(x=counts.index, y=counts.values, ax=ax, edgecolor="white", linewidth=0.7)\n\nax.set_xlabel("TYPE")\nax.set_ylabel("COUNT")\nax.tick_params(axis="x", labelrotation=45)\nax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f\'{int(x):,}\'))\n\nplt.tight_layout()\nplt.show()\n'